# AD xQTL loci table

Reproducible pipeline to generate the unified AD xQTL loci summary Excel table and deploy the Shiny explorer.

You will need 5 Metadata:
- `metadata_analysis.csv` containing all exported tables paths
- `contexts_metadata.csv` containing contexts (datasets) label mapping between the different analysis/exported tables
- `columns_metadata.tsv` containing all columns to keep for the excel sheets, with associated metadata (column width, coloring..)
- `excel_metadata.tsv` containing some meta information for the excel sheet construction
- `pattern_coloring.tsv` containing all pattern to color in the excel sheet
  
1 metadata is optional - `long_table_columns_selection.csv` to generate a long table with selected columns from the table. 

`xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5.csv.gz` generated in step III 
(in this table each row is a variant-ADlocus-Method-context-gene_name information, and so facilitate querying informations ) 



## How to Add New Data

To add a new study, cohort, or GWAS, you need to:
1. Upload files to S3 and download to cluster
2. Update the two metadata files
3. Rerun the R pipeline on BU SCC to regenerate the summary `.csv.gz` files
4. Rerun the code cell below — no changes to the code needed

---

### 1. Upload to S3 and download to cluster

```bash
# Upload
aws s3 cp new_study.exported.toploci.bed.gz \
  s3://statfungen/ftp_fgc_xqtl/analysis_result/single_context/NEW_STUDY/export/summary/context_specific/

# Download to cluster
aws s3 sync s3://statfungen/ftp_fgc_xqtl/analysis_result/single_context/NEW_STUDY/ \
  /data/analysis_result/single_context/NEW_STUDY/
```

---

### 2. Update `metadata_analysis.csv`

This file controls which input files are read for each method. Add one row per new method.
To see its current structure:
```r
mtd <- fread(file.path(out, 'metadata_analysis.csv'))
head(mtd)
```
To append a new row:
```r
mtd <- fread(file.path(out, 'metadata_analysis.csv'))
mtd <- rbind(mtd, data.table(
  `Data Type` = 'eQTL',
  Cohort      = 'NEW_COHORT',
  Modality    = 'snRNA-seq',
  context     = 'NEW_context',
  Method      = 'single_context_finemapping',   # or ColocBoost, AD_GWAS_finemapping, etc.
  Path        = 'analysis_result/single_context/NEW_STUDY/export/summary/context_specific/NEW_file.bed.gz'
), fill = TRUE)
fwrite(mtd, file.path(out, 'metadata_analysis.csv'))
```

---

### 3. Update `contexts_metadata.csv`

This file maps context IDs to their display labels and determines which Excel sheet they appear in.
To see its current structure:
```r
contexts <- fread(file.path(out, 'contexts_metadata.csv'))
head(contexts)
```
To append a new context:
```r
contexts <- fread(file.path(out, 'contexts_metadata.csv'))
contexts <- rbind(contexts, data.table(
  context        = 'NEW_context',
  context_broad  = 'snuc_NEW_eQTL',
  context_short  = 'NEW_eQTL',
  context_hi     = 'NEW_eQTL',
  context_coloc  = 'NEW_context',
  context_broad2 = 'Exc',     # which Excel sheet: Brain / Exc / Inh / Oli / OPC / Ast / Mic / Immune (bulk)
  qtl_type       = 'eQTL'
), fill = TRUE)
fwrite(contexts, file.path(out, 'contexts_metadata.csv'))
```
> ⚠️ If `context_broad2` is a completely new category not already in the Excel,
> also add it to `columns_metadata.tsv` and `excel_metadata.tsv` to create a new sheet.

---

### 4. Rerun R pipeline

From `main_text/5_AD_xQTL_genes_cis_trans/staging/gene_priorization_table/`, run in this exact order:
```bash
Rscript gene_locus_summary.R   # Sections I–III
Rscript gwas_cs_extension.R    # between Sections I and II
Rscript export_mr.R            # before MR integration in Section III
```
This regenerates:
```r
# These two files are what the code cell below reads
res_adx  <- fread(file.path(out, 'xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5.csv.gz'))
res_adfv <- fread(file.path(out, 'AD_loci_unified_cs95orColocs_Pval1e5_variant_level.csv.gz'))
```

---

### 5. Rerun the code cell below

No changes needed. The loop below automatically picks up any new `context_broad2` value
that appears in `excel_metadata.tsv`:
```r
for(cont in colsmtd[wildcard=='context_broad2']$r_name){ ... }
```


## Libraries

In [ ]:
library(data.table)
library(openxlsx)
library(stringr)

## Set up working directory

In [ ]:
out <- '/restricted/projectnb/xqtl/jaempawi/xqtl/AD_xQTL_loci/'
source(file.path(out, 'gene_prio_utils.R'))

In [2]:
options(warn = -1)

# Load inputs
res_adx   <- fread(file.path(out, 'xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5.csv.gz'))
res_adfv  <- fread(file.path(out, 'AD_loci_unified_cs95orColocs_Pval1e5_variant_level.csv.gz'))

# Load metadata
cols      <- fread(file.path(out, 'columns_metadata.tsv'))[(keep==1)]
colsmtd   <- fread(file.path(out, 'excel_metadata.tsv'))
colorsmtd <- fread(file.path(out, 'pattern_coloring.tsv'))

# Wide table
res_adxub <- WideTable(res_adx, split.by=c('context_broad2','qtl_type'))

# Filter top variants
res_adxub[, top_variants:=((max_variant_inclusion_probability>=0.1)|cV2F_rank<=5)|
            ((max_variant_inclusion_probability_rank==1|variant_rank_xqtl==1|
               (rank(pval)==1&!is.na(pval)))), by='locus_index']
res_adxubf <- res_adxub[(top_variants)][!is.na(locus_index)][variant_ID!='']

# Prep columns metadata
cols <- PrepColsMtd(cols, colsmtd, res_adxubf)

# Main sheet
wb <- CreateExcelFormat(res_adxubf, columns_mtd=cols, colors=colorsmtd)

# One sheet per broad context
for(cont in colsmtd[wildcard=='context_broad2']$r_name){
  message(cont)
  res_adxc   <- res_adx[context_broad2==cont]
  res_adxc   <- SummarizeTable(res_adxc, group.by='qtl_type')
  res_adxcub <- WideTable(res_adxc)
  
  res_adxcub[, top_variants:=((max_variant_inclusion_probability>=0.1)|cV2F_rank<=5)|
               ((max_variant_inclusion_probability_rank==1|variant_rank_xqtl==1|
                  (rank(pval)==1&!is.na(pval)))), by='ADlocus']
  res_adxcubf <- res_adxcub[(top_variants)]
  
  res_adxcubf[, gwas_significance:=ifelse(min_pval<5e-8,'genome wide',
                                    ifelse(min_pval<1e-6,'suggestive','ns'))]
  res_adxcubf[, mlog10pval:=-log10(min_pval)]
  res_adxcubf[, top_confidence:=str_extract(xQTL_effects,'CL[0-9]')]
  
  wb <- CreateExcelFormat(res_adxcubf, columns_mtd=cols, colors=colorsmtd,
                          wb=wb, sheet_name=cont)
}

# Save
saveWorkbook(wb, file.path(out, 'unified_AD_loci_xQTL_summary.xlsx'), overwrite=TRUE)

1 threads available for data.table

generating columns per context_broad2 and qtl_type

Inh eQTL

Mic eQTL

OPC eQTL

Brain eQTL

Brain mQTL

Exc eQTL

Oli eQTL

Immune (bulk) eQTL

Brain pQTL

Brain sQTL

Ast eQTL

Exc sQTL

Brain p_sQTL

Brain u_sQTL

Inh sQTL

Oli sQTL

Ast sQTL

Brain gpQTL

Brain haQTL

OPC sQTL

Mic sQTL

Ast caQTL

Mic caQTL

pvalue_sexFbeta_sexFse_sexFpvalue_sexF_interactionbeta_sexF_interactionse_sexF_interactionpvalue_sexbeta_sexse_sexpvalue_sex_interactionbeta_sex_interactionse_sex_interactionpvalue_genderbeta_genderse_genderpvalue_gender_interactionbeta_gender_interactionse_gender_interactioncoloc_indexnsnpsSNP_PPH4TADB_regionADxQTLxQTL_hitAD_hitxQTL_LAD_LPP.H0.abfPP.H1.abfPP.H2.abfL_PP.H3.abfL_PP.H4.abfresourceMAFbetahatsebetahatpnum_CSnum_IVcpipmeta_effse_meta_effmeta_pvalQQ_pvalI2gwas_studymolecular_idmethodis_imputableis_selected_methodrsq_cvpval_cvtwas_ztwas_pvaltypeblockidregion_idsusie_pipmucsfiletrans_genestrans_contextsmax_APOE4_interaction_pval.In

## Deploy Shiny app

In [ ]:
# One-time: set credentials (get token at shinyapps.io/admin/#/tokens)
# rsconnect::setAccountInfo(name='jenny-empawi', token='<TOKEN>', secret='<SECRET>')

rsconnect::deployApp(
  appDir  = shiny_app_dir,
  appName = "AD-xQTL-Explorer",
  account = "jenny-empawi"
)
